# Bug Ingestion Server (`FastHTML`)

> FastHTML / Starlette bug-report ingestion server for `solveitcrafts`.

Receives browser bug captures (description, URL, viewport, console errors, optional screenshot, and raw DOM HTML), sanitizes HTML via `fast5ever`, and persists the ticket bundle via `TicketStore` (backed by `fastgws` + `solvedrive`).

In [ ]:
#| default_exp server

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import base64, json
from typing import Any
from fasthtml.common import *
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.middleware.cors import CORSMiddleware

from craft_support.drive import TicketNotFoundError, TicketStore, validate_ticket_id
from craft_support.models import CaptureMeta, VALID_STATUSES

## Private Network Access & Screenshot Decoding

Because the in-app capture widget runs on `https://<instance>.solve.it.com` and submits captures to a local ingestion server (`http://127.0.0.1:8791`), Chromium enforces Private Network Access preflight checks. Passing both `PrivateNetworkAccessMiddleware` and `CORSMiddleware(..., allow_private_network=True)` to `FastHTML(middleware=...)` ensures `Access-Control-Allow-Private-Network: true` is present on preflight `OPTIONS` and normal HTTP responses.

In [ ]:
#| export
DEFAULT_ORIGIN_REGEX = r"^https://([a-z0-9-]+\.)*solve\.it\.com$|^https?://(127\.0\.0\.1|localhost)(:\d+)?$"

class PrivateNetworkAccessMiddleware(BaseHTTPMiddleware):
    "Attach `Access-Control-Allow-Private-Network: true` only for permitted origins or direct local requests."
    def __init__(self, app, *, allowed_origins: list[str] | None = None, origin_regex: str | None = None):
        super().__init__(app)
        self.allowed_origins = set(allowed_origins or ())
        self.origin_re = re.compile(origin_regex) if origin_regex else None

    def _origin_allowed(self, origin: str | None) -> bool:
        if not origin: return True
        if "*" in self.allowed_origins or origin in self.allowed_origins: return True
        return bool(self.origin_re and self.origin_re.match(origin))

    async def dispatch(self, request: Request, call_next):
        response = await call_next(request)
        if self._origin_allowed(request.headers.get("origin")): response.headers["Access-Control-Allow-Private-Network"] = "true"
        elif "access-control-allow-private-network" in response.headers: del response.headers["access-control-allow-private-network"]
        return response

def decode_screenshot_data_url(
    data_url: str | None, # `data:image/png;base64,...` data URL or raw base64 string
) -> bytes | None:
    "Decode a `data:image/png;base64,...` string (or raw base64) into PNG bytes."
    if not data_url: return None
    raw = data_url.strip()
    if "," in raw and raw.lower().startswith("data:"): raw = raw.split(",", 1)[1]
    try: return base64.b64decode(raw, validate=True)
    except Exception: return None

def _parse_viewport(raw: Any) -> dict[str, int]:
    if raw is None: return dict(width=0, height=0)
    if not isinstance(raw, dict): raise ValueError("viewport must be an object")
    out = {}
    for k in ("width", "height"):
        v = raw.get(k, 0)
        if isinstance(v, bool): raise ValueError(f"viewport.{k} must be an integer")
        iv = int(v or 0)
        if iv < 0: raise ValueError(f"viewport.{k} must be non-negative")
        out[k] = iv
    return out

def _parse_ticket_payload(
    body: bytes, user_agent: str, *,
    max_body_bytes: int, max_dom_bytes: int, max_screenshot_bytes: int,
) -> tuple[dict[str, Any] | None, JSONResponse | None]:
    def _err(msg: str, code: int = 400): return None, JSONResponse(dict(error=msg), status_code=code)
    if max_body_bytes and len(body) > max_body_bytes: return _err("Request payload too large", 413)
    try: payload = json.loads(body)
    except Exception: return _err("Invalid JSON body")
    if not isinstance(payload, dict): return _err("JSON body must be an object")

    description, title = str(payload.get("description") or "").strip(), str(payload.get("title") or "").strip()
    if not description and not title: return _err("Either title or description is required")

    try: viewport = _parse_viewport(payload.get("viewport"))
    except (TypeError, ValueError) as e: return _err(str(e))

    raw_errors = payload.get("console_errors")
    if raw_errors is not None and not isinstance(raw_errors, list): return _err("console_errors must be a list")

    dom_html = str(payload.get("dom_html") or "")
    if max_dom_bytes and len(dom_html.encode("utf-8", errors="replace")) > max_dom_bytes:
        return _err("DOM snapshot exceeds size limit", 413)

    screenshot_bytes = decode_screenshot_data_url(payload.get("screenshot_data_url"))
    if screenshot_bytes and max_screenshot_bytes and len(screenshot_bytes) > max_screenshot_bytes:
        return _err("Screenshot exceeds size limit", 413)

    capture = CaptureMeta(url=str(payload.get('url') or ''), viewport=viewport, user_agent=str(payload.get('user_agent') or user_agent),
        console_errors=[str(e) for e in raw_errors or []][:50], dialog_id=payload.get('dialog_id'))
    return dict(
        title=title or description.splitlines()[0][:72], description=description or title,
        reporter_notes=str(payload.get("reporter_notes") or ""), capture=capture,
        raw_dom_html=dom_html, screenshot_bytes=screenshot_bytes), None

In [ ]:
show_doc(decode_screenshot_data_url)

In [ ]:
import asyncio, httpx
from fastcore.test import test_eq
from craft_support.drive import MemoryTicketStore

In [ ]:
png = b"\x89PNG\r\n\x1a\n1234"
b64 = base64.b64encode(png).decode("ascii")
test_eq(decode_screenshot_data_url(f"data:image/png;base64,{b64}"), png)
test_eq(decode_screenshot_data_url(b64), png)
assert decode_screenshot_data_url(None) is None
assert decode_screenshot_data_url("not-valid-base64!!!") is None

## `create_app`

Exposes:
- `GET /health` — liveness probe.
- `GET /` — lightweight HTML test form for manual submissions.
- `POST /api/tickets/form` — form-encoded ticket creation.
- `POST /api/tickets` — bug capture endpoint used by the Solveit widget (JSON by default; returns an HTML status fragment and `HX-Trigger: craft-bug-saved` when `HX-Request: true`).
- `GET /api/tickets` — list tickets with optional `?status=...` filter.
- `GET /api/tickets/{ticket_id}` — fetch a single ticket's JSON metadata.

In [ ]:
#| export
def create_app(
    store: TicketStore, # Ticket storage backend (`DriveTicketStore`, `LocalTicketStore`, or `MemoryTicketStore`)
    *,
    allowed_origins: list[str] | None = None, # Explicit CORS origins (defaults to Solveit + localhost regex)
    max_body_bytes: int = 5_000_000, # Maximum JSON request body size in bytes
    max_dom_bytes: int = 2_000_000, # Maximum raw DOM HTML size in bytes
    max_screenshot_bytes: int = 2_000_000, # Maximum decoded screenshot size in bytes
) -> FastHTML:
    "Create the FastHTML ingestion ASGI application."
    origin_regex = None if allowed_origins is not None else DEFAULT_ORIGIN_REGEX
    middleware = [
        Middleware(PrivateNetworkAccessMiddleware, allowed_origins=allowed_origins, origin_regex=origin_regex),
        Middleware(
            CORSMiddleware, allow_origins=allowed_origins or [], allow_origin_regex=origin_regex,
            allow_credentials=False, allow_methods=["GET", "POST", "PATCH", "OPTIONS"],
            allow_headers=["*"], expose_headers=["HX-Trigger"], allow_private_network=True)]
    app = FastHTML(title="solveitcrafts Bug Ingestion", middleware=middleware)
    app.state.ticket_store = store
    rt = app.route

    @rt("/health")
    async def health() -> JSONResponse: return JSONResponse(dict(ok=True, service="craft-support"))

    @rt("/api/tickets/form")
    async def ingest_form(
        request: Request, title: str = "", description: str = "", reporter_notes: str = "",
    ) -> JSONResponse:
        title, description, reporter_notes = title.strip(), description.strip(), reporter_notes.strip()
        if not description: return JSONResponse(dict(error="description is required"), status_code=400)
        ticket, san = await store.create_ticket(title=title or description[:72], description=description, reporter_notes=reporter_notes,
            capture=CaptureMeta(url=str(request.url)), raw_dom_html='')
        return JSONResponse(dict(
            id=ticket.id, status=ticket.status, folder_url=ticket.drive.folder_url,
            sanitized_bytes=san.sanitized_bytes, redacted_fields=san.redacted_fields), status_code=201)

    @rt("/")
    async def home():
        tickets = await store.list_tickets()
        return Titled(
            "solveitcrafts Support Ingestion",
            P(f"Active tickets stored: {len(tickets)}"),
            Div(
                H2("Submit a Test Bug Report"),
                Form(
                    Label("Title", Input(name="title", placeholder="Short summary", required=True)),
                    Label("Description", Textarea(name="description", placeholder="What went wrong?", required=True)),
                    Label("Reporter notes", Textarea(name="reporter_notes", placeholder="Optional reproduction steps")),
                    Button("Submit ticket", type="submit"),
                    method="post", action=ingest_form)))

    @rt("/api/tickets", methods="post")
    async def ingest_ticket(request: Request):
        is_htmx = request.headers.get("hx-request", "").lower() == "true"
        ctype = (request.headers.get("content-type") or "").lower()
        if "application/json" in ctype or not is_htmx:
            raw_body = await request.body()
        else:
            form = await request.form()
            try: errs = json.loads(str(form.get("console_errors") or "[]"))
            except Exception: errs = []
            payload = dict(
                title=str(form.get("title") or ""), description=str(form.get("description") or ""),
                reporter_notes=str(form.get("reporter_notes") or ""), url=str(form.get("url") or ""),
                viewport=dict(width=form.get("viewport_width") or 0, height=form.get("viewport_height") or 0),
                user_agent=str(form.get("user_agent") or ""), dialog_id=form.get("dialog_id") or None,
                console_errors=errs, dom_html=str(form.get("dom_html") or ""),
                screenshot_data_url=str(form.get("screenshot_data_url") or "") or None)
            raw_body = json.dumps(payload).encode("utf-8")
        parsed, err = _parse_ticket_payload(raw_body, request.headers.get('user-agent', ''), max_body_bytes=max_body_bytes,
            max_dom_bytes=max_dom_bytes, max_screenshot_bytes=max_screenshot_bytes)
        if err:
            if not is_htmx: return err
            msg = json.loads(err.body).get("error", "Invalid submission")
            return HTMLResponse(to_xml(Span(f"Failed to submit: {msg}", cls="craft-bug-status err")), status_code=200)
        ticket, san = await store.create_ticket(**parsed)
        if is_htmx:
            suffix = f" ({ticket.drive.folder_url})" if ticket.drive.folder_url else ""
            return HTMLResponse(
                to_xml(Span(f"Saved ticket {ticket.id}{suffix}", cls="craft-bug-status ok")),
                status_code=201, headers={"HX-Trigger": "craft-bug-saved"})
        return JSONResponse(dict(
            id=ticket.id, status=ticket.status, folder_id=ticket.drive.folder_id, folder_url=ticket.drive.folder_url,
            dom_url=ticket.drive.dom_url, screenshot_url=ticket.drive.screenshot_url,
            sanitized_bytes=san.sanitized_bytes, stripped_tags=san.stripped_tags, redacted_fields=san.redacted_fields,
        ), status_code=201)

    @rt("/api/tickets", methods="get")
    async def list_tickets(status: str | None = None) -> JSONResponse:
        if status and status not in VALID_STATUSES: return JSONResponse(dict(error=f"Invalid status {status!r}"), status_code=400)
        tickets = await store.list_tickets(status=status)  # type: ignore[arg-type]
        return JSONResponse(dict(tickets=[t.to_dict() for t in tickets]))

    @rt("/api/tickets/{ticket_id}", methods="get")
    async def get_ticket(ticket_id: str) -> JSONResponse:
        try: tid = validate_ticket_id(ticket_id)
        except ValueError as e: return JSONResponse(dict(error=str(e)), status_code=400)
        try: ticket = await store.get_ticket(tid)
        except TicketNotFoundError: return JSONResponse(dict(error=f"Ticket {tid} not found"), status_code=404)
        return JSONResponse(ticket.to_dict())

    return app

In [ ]:
show_doc(create_app)

In [ ]:
async def _test_server():
    store = MemoryTicketStore()
    app = create_app(store)
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://testserver") as client:
        preflight = await client.options("/api/tickets", headers={
            "Origin": "https://example.solve.it.com",
            "Access-Control-Request-Method": "POST",
            "Access-Control-Request-Private-Network": "true"})
        test_eq((preflight.status_code, preflight.headers.get("access-control-allow-private-network")), (200, "true"))

        health = await client.get("/health")
        test_eq((health.status_code, health.json()["ok"], health.headers.get("access-control-allow-private-network")), (200, True, "true"))
        home_res = await client.get("/")
        test_eq((home_res.status_code, home_res.text.count("<h1")), (200, 1))
        assert 'action="/api/tickets/form"' in home_res.text

        r = await client.post("/api/tickets", json=dict(
            title="Hotkey conflict", description="Mod+B does not toggle tree",
            url="https://solve.it.com/dialog/42", viewport=dict(width=1280, height=720),
            dom_html='<script>evil()</script><input type="password" value="hunter2"><p>Bug</p>',
            screenshot_data_url=f"data:image/png;base64,{b64}"))
        test_eq(r.status_code, 201)
        tid = r.json()["id"]
        dom = await store.get_dom(tid)
        assert "<script" not in dom.lower() and "hunter2" not in dom
        test_eq(await store.get_screenshot(tid), png)

        rf = await client.post("/api/tickets/form", data=dict(title="Form bug", description="Submitted via HTML form"))
        test_eq(rf.status_code, 201)

        rh = await client.post("/api/tickets", data=dict(
            title="HTMX bug", description="Submitted via hx-post",
            viewport_width="1280", viewport_height="720", console_errors='["err1"]'),
            headers={"HX-Request": "true"})
        test_eq((rh.status_code, rh.headers.get("hx-trigger")), (201, "craft-bug-saved"))
        assert "Saved ticket " in rh.text and 'craft-bug-status ok' in rh.text

        test_eq(len((await client.get("/api/tickets")).json()["tickets"]), 3)
        test_eq((await client.get(f"/api/tickets/{tid}")).status_code, 200)
        test_eq((await client.get("/api/tickets/TKT-MISSING")).status_code, 404)
        test_eq((await client.get("/api/tickets?status=invalid")).status_code, 400)
        evil = await client.options("/api/tickets", headers={
            "Origin": "https://evil.example.com",
            "Access-Control-Request-Method": "POST",
            "Access-Control-Request-Private-Network": "true"})
        assert evil.headers.get("access-control-allow-private-network") is None
        test_eq((await client.post("/api/tickets", json=dict(title="Bad vp", viewport=dict(width="abc")))).status_code, 400)
        test_eq((await client.get("/api/tickets/bad..id")).status_code, 400)
        assert (await client.get("/api/tickets/..%2Fescape")).status_code in (400, 404)

    small_app = create_app(store, max_body_bytes=64)
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=small_app), base_url="http://testserver") as client:
        test_eq((await client.post("/api/tickets", json=dict(title="Too big", description="x" * 200))).status_code, 413)

await _test_server()